# Supplemental: Querying Wikipedia with the MediaWiki API

**CS1090A / AC209A — Module 1**

This notebook shows how to pull structured data from Wikipedia **directly through
the official [MediaWiki API](https://www.mediawiki.org/wiki/API:Main_page)** using
`requests` — the same skill you already use for scraping, pointed at a clean JSON
API instead of raw HTML.

Why not the `wikipedia` PyPI package? It is unmaintained, scrapes the rendered
site, and breaks under rate limits. The MediaWiki API is stable, gives you a
**permanent numeric `pageid`**, and is the reproducible way to do this.

**Supports HW1 Q6–Q8:** resolve each film to a `pageid`, cache its page HTML to
disk, and extract a field (e.g. language) from the infobox.

## What you'll build

1. **Search** for a page and get its stable `pageid`.
2. **Fetch** a page's rendered HTML and **cache it to disk** (guarded fetch).
3. **Parse** a value out of the infobox with BeautifulSoup.
4. Apply the pipeline across a small **DataFrame of films**.
5. (Optional / grad) Speed it up with **polite concurrency**.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os, json, time

## Setup: one endpoint, and being a polite citizen

Every request goes to the same endpoint. Wikipedia asks that you send a
descriptive `User-Agent` identifying your project so they can contact you if your
script misbehaves. We also reuse a single `Session` for connection pooling, and
our `api_get` helper **retries politely** if Wikipedia throttles us (HTTP 429).

In [2]:
API = "https://en.wikipedia.org/w/api.php"

session = requests.Session()
session.headers.update({
    "User-Agent": "CS1090A-course-example/1.0 (Harvard CS1090A; instructor contact)"
})

def api_get(params, max_retries=4):
    """GET the MediaWiki API, retrying politely on rate-limit / overload."""
    params = {"format": "json", "formatversion": 2, **params}
    for attempt in range(max_retries):
        r = session.get(API, params=params, timeout=20)
        if r.status_code in (429, 503):           # rate-limited / overloaded
            wait = int(r.headers.get("Retry-After", 2 ** attempt))
            print(f"  throttled ({r.status_code}); waiting {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()                          # give up after retries

## 1. Search → `pageid`

The `query`/`search` module does a full-text search and returns candidate pages.
Build the query from what you know about the film (title + year + the word
`"film"` disambiguates from soundtracks, novels, etc.).

In [3]:
def search(query, limit=3):
    """Return a list of (pageid, title) search hits, best first."""
    data = api_get({"action": "query", "list": "search",
                    "srsearch": query, "srlimit": limit})
    return [(h["pageid"], h["title"]) for h in data["query"]["search"]]

search("Big Trouble in Little China 1986 film")

[(697113, 'Big Trouble in Little China'),
 (22535247, 'Big Trouble in Little China (soundtrack)'),
 (22614951, 'Big Trouble in Little China (video game)')]

The top hit is usually the film itself, but notice the runners-up (a soundtrack, a
video game). A good heuristic: take the first hit, but in real code you might
verify the page really is a film (see the exercises).

In [4]:
def get_pageid(title, year=None):
    """Resolve a film title (+ optional year) to a stable pageid, or None."""
    query = f"{title} {year} film" if year else f"{title} film"
    hits = search(query, limit=1)
    return hits[0][0] if hits else None

pid = get_pageid("Big Trouble in Little China", 1986)
pid

697113

## 1b. Mind your request budget: batch exact-title lookups

Search costs **one request per film**. For a thousand films that's a thousand
requests — slow for you, rude to Wikipedia, and likely to get you throttled.

But film articles follow strong naming conventions — `"Title (YEAR film)"`,
`"Title (film)"`, or plain `"Title"` — and the API's `action=query&titles=`
form checks up to **50 exact titles in a single request** (add `redirects=1`
so `"Star Wars"` finds the article it redirects to). Resolve the easy ~90%
this way, and save search for the stragglers.

In [5]:
def batch_lookup(titles):
    """Exact-title lookup: {queried title -> pageid}, 50 titles per request."""
    found = {}
    for i in range(0, len(titles), 50):
        chunk = titles[i:i + 50]
        data = api_get({"action": "query", "titles": "|".join(chunk),
                        "redirects": 1})
        redirects = {r["from"]: r["to"] for r in data["query"].get("redirects", [])}
        by_title = {p["title"]: p.get("pageid")
                    for p in data["query"]["pages"] if not p.get("missing")}
        for q in chunk:
            target = redirects.get(q, q)
            if target in by_title:
                found[q] = by_title[target]
    return found

# One request checks all of these. Note which patterns hit: there is no
# "Big Trouble in Little China (1986 film)" redirect -- the bare title is the
# article -- while Kurosawa's Stray Dog lives at the '(film)' pattern.
batch_lookup(["Big Trouble in Little China (1986 film)",
              "Big Trouble in Little China",
              "Stray Dog (film)",
              "Jaws"])

{'Big Trouble in Little China': 697113,
 'Stray Dog (film)': 867905,
 'Jaws': 1350252}

Pattern for a whole dataset: try `"{title} ({year} film)"`, then
`"{title} (film)"`, then `"{title}"` (most-specific first — the bare title
can hit a same-named non-film page), and fall back to `search()` only for
films none of the patterns resolved. ~100× fewer requests than search-per-film.

## 1c. Batch the *content* too: wikitext, 50 pages per request

Batching isn't just for lookups. The `prop=revisions` module returns a page's
**wikitext source** — the raw markup, infobox included — and it accepts up to
**50 `pageids` per request**. For fetching many pages' content, this is the
only polite option: per-page endpoints (like `action=parse`, §2 below) get you
throttled long before a thousand requests complete.

**This is the endpoint HW1 Q7 requires** — the infobox fields you'll extract
in Q8 (e.g. `| language = [[English language|English]]`) are right in the source.

In [6]:
def batch_wikitext(pageids):
    """Fetch up to 50 pages' wikitext in ONE request: {pageid: text}."""
    data = api_get({"action": "query",
                    "pageids": "|".join(str(p) for p in pageids),
                    "prop": "revisions", "rvprop": "content",
                    "rvslots": "main"})
    return {p["pageid"]: p["revisions"][0]["slots"]["main"]["content"]
            for p in data["query"]["pages"] if p.get("revisions")}

pages = batch_wikitext([697113, 867905])   # one request, two pages
text = pages[697113]
print(f"{len(pages)} pages, {len(text):,} chars each-ish. The infobox params:")
import re
print(re.search(r"\| *language *= *.*", text).group(0))

2 pages, 58,959 chars each-ish. The infobox params:
| language       = English


Wikitext is markup, not rendered HTML — extracting a field means a
line-anchored regex on the template parameter plus cleanup of piped links and
`<ref>`s (messier than the HTML infobox below, but you paid ~26 requests for
1,300 pages instead of 1,300). Budget, then choose.

## 2. Fetch rendered page HTML — and cache it to disk

The `parse` module returns a page's rendered HTML — nicest to work with, but
**one request per page**: right for the handful of pages here, wrong at
HW1 scale (use §1c's batched wikitext there). We wrap it in a **guarded
fetch**: if we already have the HTML on disk we read the file; otherwise we hit
the API and save it, keyed by the stable `pageid`.

This is the heart of a **reproducible pipeline**: re-running the notebook does not
re-download anything, and a collaborator (or grader) with your cache reproduces
your results without touching the network.

In [7]:
CACHE_DIR = "data/wiki"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_page_html(pageid):
    """Return a page's HTML, reading from / writing to the on-disk cache."""
    if pageid is None:
        return None
    path = os.path.join(CACHE_DIR, f"{pageid}.html")
    if os.path.isfile(path):                      # guarded: skip the network
        with open(path, encoding="utf-8") as f:
            return f.read()
    data = api_get({"action": "parse", "pageid": pageid, "prop": "text"})
    html = data["parse"]["text"]
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return html

html = get_page_html(pid)
print(f"{len(html):,} chars of HTML; cached at {CACHE_DIR}/{pid}.html")

187,388 chars of HTML; cached at data/wiki/697113.html


**Reproducible vs. brittle.** The `pageid` is permanent, so your *identifier* is
stable. The page *content*, though, drifts as editors update articles — so caching
the HTML is what actually makes your extracted data reproducible. (This is exactly
the kind of brittleness HW1 asks you to reflect on.)

## 3. Extract a field from the infobox

Most film articles have a standardized **infobox** (the table at the top right).
Its rows are `<th>` label / `<td>` value pairs — easy to target with BeautifulSoup.

In [8]:
def infobox_field(html, label):
    """Return the infobox value whose row label starts with `label`, or None."""
    soup = BeautifulSoup(html, "html.parser")
    for th in soup.select("table.infobox th"):
        if th.get_text(strip=True).startswith(label):
            td = th.find_next("td")
            return td.get_text(", ", strip=True) if td else None
    return None

print("Language:", infobox_field(html, "Language"))
# Infobox values are often messy in the wild — note the stray citation marker:
print("Running time:", infobox_field(html, "Running time"))

Language: English


Running time: 99 minutes, [, 1, ]


## 4. Put it together over a small DataFrame

This is the HW1 pattern in miniature: start from a frame of films, resolve each to
a `wiki_id`, cache the HTML, and pull a new column out of the infobox.

In [9]:
films = pd.DataFrame([
    {"title": "Big Trouble in Little China", "year": 1986},
    {"title": "Stray Dog", "year": 1949},
    {"title": "Young Frankenstein", "year": 1974},
    {"title": "2001: A Space Odyssey", "year": 1968},
    {"title": "Ghost Dog: The Way of the Samurai", "year": 1999},
])

films["wiki_id"] = [get_pageid(t, y) for t, y in zip(films.title, films.year)]
films["language"] = [
    infobox_field(get_page_html(pid), "Language") if pd.notna(pid) else None
    for pid in films.wiki_id
]
films

,title,year,wiki_id,language
0,Big Trouble in Little China,1986,697113,English
1,Stray Dog,1949,867905,Japanese
2,Young Frankenstein,1974,442647,English
3,2001: A Space Odyssey,1968,23941708,English
4,Ghost Dog: The Way of the Samurai,1999,471352,"English, French"


## 5. (Optional / required for grads) Polite concurrency

The calls above are blocking I/O, so a small thread pool speeds things up a lot.
Keep concurrency **low** (2–4 workers) to stay polite — you are a guest on
Wikipedia's servers. Because `get_page_html` caches to disk, re-runs stay fast and
hit the network only for pages you don't yet have.

In [10]:
from concurrent.futures import ThreadPoolExecutor

def cache_all(pageids, max_workers=2):
    """Warm the on-disk cache for many pageids using a small thread pool."""
    ids = [p for p in pageids if pd.notna(p)]
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        list(ex.map(get_page_html, ids))

t0 = time.time()
cache_all(films.wiki_id)
print(f"warmed cache in {time.time() - t0:.2f}s (fast on re-run — already cached)")

warmed cache in 0.01s (fast on re-run — already cached)


## Recap & exercises

You now have the full HW1 enrichment pipeline: **search → `pageid` → cached HTML
→ infobox field**, all through the stable MediaWiki API and reproducible via the
on-disk cache.

**Try it:**
1. Add a `budget` (or `Box office`) column using `infobox_field`. What makes these
   values messy to work with, and how would you clean them?
2. Harden `get_pageid`: fetch each candidate's categories
   (`action=query&prop=categories`) and prefer a hit whose categories mention
   `"films"`. Does it fix any mismatches?
3. What happens for a film with no Wikipedia page? Make the pipeline degrade
   gracefully (it should already return `None` — confirm where).

---
## Pulling a field out of the wikitext — HW1 Q8.1

Question 7 leaves you a directory of **wikitext**, not rendered HTML, so there is no
BeautifulSoup tree to walk. The infobox is a template whose parameters sit **one per
line**:

```
{{Infobox film
| name        = Nosferatu
| language    = [[German language|German]]
| runtime     = 94 minutes
}}
```

One line per parameter is the thing to exploit. A regex anchored to the **start of a
line** finds the parameter you want without matching the same word elsewhere in a
40,000-character article.

`^` normally means "start of the string". **`re.MULTILINE` makes it mean "start of any
line"** — which is what turns the layout above into something you can match.

Compare the two directly.

In [ ]:
import re

article = """{{Infobox film
| name     = Nosferatu
| language = [[German language|German]]
| runtime  = 94 minutes
}}
The language of the intertitles was later changed for the American release.
"""

# Without MULTILINE, ^ only matches the very start of the string -> no match at all.
print("plain     :", re.findall(r"^\|\s*language\s*=\s*(.+)$", article))

# With MULTILINE, ^ and $ match at every line boundary -> the infobox line, and only it.
print("MULTILINE :", re.findall(r"^\|\s*language\s*=\s*(.+)$", article, flags=re.MULTILINE))

The second one found the field and **ignored the prose sentence** that also says
"language". That is the whole reason to anchor.

`re.M` is the same flag, spelled shorter; you will see both.

**Then comes the cleaning**, which is the part that actually takes the time. Wikitext
values carry piped links `[[a|b]]`, `{{Plainlist}}` wrappers, `<ref>` citations and HTML
comments. Extracting the field and *getting a clean value* are two different jobs — Q8.1
asks you to describe the messiness you hit, so keep notes as you go.

In [ ]:
def clean_value(raw):
    """Good enough for HW1 Q8: strip the wrappers, keep the display text."""
    raw = re.sub(r"<ref[^>]*>.*?</ref>", "", raw, flags=re.S)   # citations
    raw = re.sub(r"<!--.*?-->", "", raw, flags=re.S)            # comments
    raw = re.sub(r"\[\[[^\]|]*\|([^\]]*)\]\]", r"\1", raw)      # [[target|shown]] -> shown
    raw = re.sub(r"\[\[([^\]]*)\]\]", r"\1", raw)              # [[plain]]        -> plain
    raw = re.sub(r"\{\{[^}]*\}\}", " ", raw)                    # {{templates}}
    return re.sub(r"\s+", " ", raw).strip(" ,;")

for field in re.findall(r"^\|\s*language\s*=\s*(.+)$", article, flags=re.MULTILINE):
    print(repr(clean_value(field)))